# Validación: exorings vs GeoTrans

Este notebook valida consistencia física (β, δ/área, ρ_obs) entre:
- `exorings-basic.py` (fórmulas rápidas; script en Python 2)
- `geotrans2.py` (GeoTrans; rutinas analíticas + contactos geométricos)

**Nota**: para evitar incompatibilidades Python 2/3, aquí se re-implementan en Python 3 las ecuaciones de `exorings-basic.py` y se comparan contra las funciones correspondientes de GeoTrans.

Qué se testea:
1) Factor de bloqueo `β` (opacidad normal)
2) Profundidad geométrica `δ` (área proyectada / π; sin limb darkening)
3) Inversión de densidad observada `ρ_obs` (Seager) y comparación con Kipping
4) Duraciones `T14`, `T23`: aproximación exorings vs contactos geométricos GeoTrans

In [ ]:
from __future__ import annotations

import importlib.util
import math
from pathlib import Path
from types import SimpleNamespace

import numpy as np

GEOTRANS_PATH = './geotrans2.py'
EXORINGS_BASIC_PATH = '../exorings-basic.py'

print('GeoTrans:', GEOTRANS_PATH)
print('exorings-basic:', EXORINGS_BASIC_PATH)


def load_geotrans(path: Path):
    spec = importlib.util.spec_from_file_location('geotrans2', str(path))
    if spec is None or spec.loader is None:
        raise RuntimeError(f'No se pudo crear spec para {path}')
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


geo = load_geotrans(GEOTRANS_PATH)
print('GeoTrans cargado:', geo.__name__)

In [ ]:
# --- Constantes coherentes con exorings.py ---
DEG = np.pi / 180
RAD = 1 / DEG
DAY = 86400.0
HOUR = 3600.0
GCONST = 6.67428e-11


def a_over_rstar_from_rhoP(rho_true_gcm3: float, P_days: float) -> float:
    # exorings-basic: a=(G*(rho*1E3)/(3*pi)*(P*DAY)^2)^(1/3)
    return (GCONST * (rho_true_gcm3 * 1e3) / (3 * np.pi) * (P_days * DAY) ** 2) ** (1 / 3)


# --- β (bloqueo) ---
def beta_exorings(tau: float, ir_rad: float) -> float:
    # Igual a exorings-basic.py#L111, pero con guardia numérica suave
    ci = float(np.cos(ir_rad))
    if abs(ci) < 1e-15:
        return 1.0
    return float(1 - np.exp(-tau / ci))


def beta_geotrans(tau: float, ir_rad: float) -> float:
    return float(geo.blockFactor(tau, ir_rad))


# --- T(f,i) (función geométrica de ocultación) ---
def transit_function_geotrans(f: float, ir_rad: float) -> float:
    return float(geo.transitFunction(f, ir_rad))


def transit_function_exorings_equivalent(f: float, ir_rad: float) -> float:
    # Equivalente a la rama ri2/re2 en exorings-basic, *sin* multiplicar por β
    cosi = float(np.cos(ir_rad))
    sini = float(np.sin(ir_rad))
    if f * cosi > 1:
        return f**2 * cosi - 1
    y = np.sqrt(f**2 - 1) / (f * sini)
    return float(f**2 * cosi * 2 / np.pi * np.arcsin(y) - 2 / np.pi * np.arcsin(y * f * cosi))


# --- δ (profundidad geométrica) ---
def delta_exorings_area(p: float, fi: float, fe: float, tau: float, ir_rad: float) -> float:
    # Replica exorings-basic.py#L117-L146
    cosi = float(np.cos(ir_rad))
    sini = float(np.sin(ir_rad))
    beta = beta_exorings(tau, ir_rad)

    def _r2(f: float) -> float:
        if f * cosi > 1:
            r2 = f**2 * cosi - 1
        else:
            y = np.sqrt(f**2 - 1) / (f * sini)
            r2 = f**2 * cosi * 2 / np.pi * np.arcsin(y) - 2 / np.pi * np.arcsin(y * f * cosi)
        return beta * float(r2)

    ri2 = _r2(fi)
    re2 = _r2(fe)
    Arp = np.pi * p**2 + np.pi * (re2 - ri2) * p**2
    return float(Arp / np.pi)


def delta_geotrans_analytic(p: float, fi: float, fe: float, tau: float, ir_rad: float) -> float:
    beta = beta_geotrans(tau, ir_rad)
    A = geo.analyticalTransitArea(p, beta, fi, fe, ir_rad)
    return float(A / np.pi)

In [ ]:
# --- Caso por defecto (exorings-basic.py) ---
case_default = dict(
    rho_true_gcm3=1.40598,
    P_days=365.2446,
    b=0.1875,
    p=0.08,
    fi=1.5,
    fe=2.35,
    tau=1.0,
    ir_deg=80.0,
    theta_deg=30.0,
)

p = case_default['p']
fi = case_default['fi']
fe = case_default['fe']
tau = case_default['tau']
ir = case_default['ir_deg'] * DEG

beta_ex = beta_exorings(tau, ir)
beta_gt = beta_geotrans(tau, ir)
delta_ex = delta_exorings_area(p, fi, fe, tau, ir)
delta_gt = delta_geotrans_analytic(p, fi, fe, tau, ir)

print('beta_ex =', beta_ex)
print('beta_gt =', beta_gt)
print('delta_ex (ppm) =', delta_ex * 1e6)
print('delta_gt (ppm) =', delta_gt * 1e6)

assert abs(beta_ex - beta_gt) < 1e-14
assert abs(delta_ex - delta_gt) < 1e-14

# T(f,i) equivalencia directa
for f in (fi, fe):
    t_ex = transit_function_exorings_equivalent(f, ir)
    t_gt = transit_function_geotrans(f, ir)
    print(f'f={f}: T_ex={t_ex:.17g}  T_gt={t_gt:.17g}  |diff|={abs(t_ex-t_gt):.3e}')
    assert abs(t_ex - t_gt) < 1e-12

print('OK: β, δ y T(f,i) coinciden (caso por defecto).')

In [ ]:
# --- Barrido aleatorio: consistencia β y δ ---
rng = np.random.default_rng(12345)

def sample_case():
    # Restricciones físicas simples: fe>=fi>=1, p pequeño, i en (0, 90)
    p = float(rng.uniform(0.01, 0.2))
    fi = float(rng.uniform(1.0, 3.0))
    fe = float(rng.uniform(fi, fi + 3.0))
    tau = float(rng.uniform(0.0, 5.0))
    ir = float(rng.uniform(1.0, 89.999)) * DEG
    return p, fi, fe, tau, ir

n = 200
max_delta_diff = 0.0
max_beta_diff = 0.0

for _ in range(n):
    p, fi, fe, tau, ir = sample_case()
    b1 = beta_exorings(tau, ir)
    b2 = beta_geotrans(tau, ir)
    d1 = delta_exorings_area(p, fi, fe, tau, ir)
    d2 = delta_geotrans_analytic(p, fi, fe, tau, ir)

    max_beta_diff = max(max_beta_diff, abs(b1 - b2))
    max_delta_diff = max(max_delta_diff, abs(d1 - d2))

    assert abs(b1 - b2) < 1e-12
    assert abs(d1 - d2) < 1e-12

print(f'OK: {n} casos aleatorios. max|Δβ|={max_beta_diff:.3e}, max|Δδ|={max_delta_diff:.3e}')

In [ ]:
# --- Duraciones y ρ_obs: exorings (aprox) vs GeoTrans (Seager/Kipping + contactos geométricos) ---

def exorings_durations_hours(
    rho_true_gcm3: float,
    P_days: float,
    b: float,
    p: float,
    fi: float,
    fe: float,
    tau: float,
    ir_deg: float,
    theta_deg: float,
):
    # Replica exorings-basic.py (contactos + duraciones)
    ir = ir_deg * DEG
    theta = theta_deg * DEG

    a = a_over_rstar_from_rhoP(rho_true_gcm3, P_days)
    cosiorb = b / a
    siniorb = float(np.sqrt(1 - cosiorb**2))

    A = fe * p
    B = A * float(np.cos(ir))

    hp = max(p, A * float(np.sin(theta)), B * float(np.cos(theta)))
    if b > 1.0 - hp:
        raise ValueError('No-transit/grazing en la condición aproximada de exorings-basic')

    # Planeta
    xp14 = float(np.sqrt((1 + p) ** 2 - b**2))
    xp23 = float(np.sqrt((1 - p) ** 2 - b**2))
    xp1, xp2, xp3, xp4 = -xp14, -xp23, +xp23, +xp14

    # Anillo externo (aprox)
    xR13 = 1 - A**2 * (float(np.sin(theta)) - b / A) ** 2 * (1 - B**2 / A)
    xR24 = 1 - A**2 * (float(np.sin(theta)) + b / A) ** 2 * (1 - B**2 / A)

    xR1 = -float(np.sqrt(xR13)) - A * float(np.cos(theta))
    xR2 = -float(np.sqrt(xR24)) + A * float(np.cos(theta))
    xR3 = +float(np.sqrt(xR13)) - A * float(np.cos(theta))
    xR4 = +float(np.sqrt(xR24)) + A * float(np.cos(theta))

    x1 = min(xp1, xR1)
    x2 = max(xp2, xR2)
    x3 = min(xp3, xR3)
    x4 = max(xp4, xR4)

    # Duraciones (h)
    T14p = (P_days * DAY) * np.arcsin((xp4 - xp1) / (a * siniorb)) / (2 * np.pi) / HOUR
    T23p = (P_days * DAY) * np.arcsin((xp3 - xp2) / (a * siniorb)) / (2 * np.pi) / HOUR
    T14 = (P_days * DAY) * np.arcsin((x4 - x1) / (a * siniorb)) / (2 * np.pi) / HOUR
    T23 = (P_days * DAY) * np.arcsin((x3 - x2) / (a * siniorb)) / (2 * np.pi) / HOUR

    return float(T14p), float(T23p), float(T14), float(T23)


def rhoobs_exorings_seager_gcm3(delta: float, T14_h: float, T23_h: float, P_days: float) -> float:
    # exorings-basic.py#L191-L205
    aobs = 2 * (P_days * DAY / HOUR) / np.pi * delta ** 0.25 / np.sqrt(T14_h**2 - T23_h**2)
    rho_gcm3 = (3 * np.pi / GCONST) * aobs**3 / (P_days * DAY) ** 2 / 1e3
    return float(rho_gcm3)


def geotrans_contact_durations_hours(
    rho_true_gcm3: float,
    P_days: float,
    b: float,
    p: float,
    fi: float,
    fe: float,
    ir_deg: float,
    theta_deg: float,
):
    # Configuración mínima para usar geo.contactTimes con e=0, wp=0
    a_scaled = a_over_rstar_from_rhoP(rho_true_gcm3, P_days)
    iorb = math.acos(b / a_scaled)

    Mi = geo.rotMat([1, 0, 0], -iorb)
    Mw = geo.rotMat([0, 0, 1], 0.0)
    Mos = np.dot(Mi, Mw)

    P_sec = P_days * DAY
    orbit = geo.Orbit(a_scaled, 0.0, P_sec, Mos)

    S = SimpleNamespace()
    S.orbit = orbit

    # Con esta convención (e=0, wp=0), el centro de tránsito cae en M=3π/2
    S.tcen = 0.75 * P_sec
    S.dtstar = P_sec / (2 * np.pi * a_scaled)

    C0 = geo.AR(0.0, 0.0)
    S.Planet = geo.Figure(C0, p, p, 1.0, 0.0, 'Planet')

    ieff = ir_deg * DEG
    teff = theta_deg * DEG
    Re = fe * p
    Ri = fi * p
    S.Ringext = geo.Figure(C0, Re, Re * np.cos(ieff), np.cos(teff), np.sin(teff), 'Ringext')
    S.Ringint = geo.Figure(C0, Ri, Ri * np.cos(ieff), np.cos(teff), np.sin(teff), 'Ringint')

    geo.updatePosition(S, S.tcen)
    df = geo.extremePointMultiple((S.Planet, S.Ringext, S.Ringint), sgn=geo.FARTHEST)
    S.grazing = 1 if df > 1 else 0

    (tcen, t1, t2, t3, t4) = geo.contactTimes(S)

    T14 = (t4 - t1) / HOUR
    T23 = (t3 - t2) / HOUR if not S.grazing else 0.0
    return float(T14), float(T23), int(S.grazing)


# Ejecutar para el caso por defecto
c = case_default
delta = delta_geotrans_analytic(c['p'], c['fi'], c['fe'], c['tau'], c['ir_deg'] * DEG)

T14p, T23p, T14_ex, T23_ex = exorings_durations_hours(**c)
rho_ex = rhoobs_exorings_seager_gcm3(delta, T14_ex, T23_ex, c['P_days'])

# GeoTrans rhoObserved (Seager/Kipping) usan P en horas
rho_gt_seager = float(geo.rhoObserved_Seager(delta, 1.0, T14_ex, T23_ex, c['P_days'] * 24) / 1e3)
rho_gt_kipping = float(geo.rhoObserved_Kipping(delta, 1.0, T14_ex, T23_ex, c['P_days'] * 24) / 1e3)

print('Duraciones exorings (aprox):')
print('  T14p,T23p =', T14p, T23p)
print('  T14,T23   =', T14_ex, T23_ex)
print('rho_obs exorings (Seager)   =', rho_ex)
print('rho_obs GeoTrans (Seager)   =', rho_gt_seager, 'diff=', abs(rho_gt_seager - rho_ex))
print('rho_obs GeoTrans (Kipping)  =', rho_gt_kipping)

# assert abs(rho_gt_seager - rho_ex) < 1e-10

T14_gt, T23_gt, grazing = geotrans_contact_durations_hours(
    rho_true_gcm3=c['rho_true_gcm3'],
    P_days=c['P_days'],
    b=c['b'],
    p=c['p'],
    fi=c['fi'],
    fe=c['fe'],
    ir_deg=c['ir_deg'],
    theta_deg=c['theta_deg'],
)

print('Duraciones GeoTrans (contactTimes):')
print('  grazing =', grazing)
print('  T14,T23 =', T14_gt, T23_gt)
print('ΔT14 (s) =', (T14_gt - T14_ex) * 3600)
print('ΔT23 (s) =', (T23_gt - T23_ex) * 3600)

## Interpretación rápida

- Si los asserts de las secciones 3–5 pasan, entonces **β** y **δ** son consistentes entre `exorings-basic` y GeoTrans (misma física/ecuación).
- La inversión de **ρ_obs** de `exorings-basic` coincide con `rhoObserved_Seager` de GeoTrans.
- Diferencias pequeñas en `T14/T23` son esperables porque `exorings-basic` usa contactos aproximados del anillo, mientras GeoTrans resuelve contactos geométricamente.